In [ ]:
import geopandas as gpd
from pathlib import Path
import fiona
import matplotlib.pyplot as plt

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
networks_folder = base_path / "Processed_data/networks"

# Define the output directory path
networks_catchments_intersections = base_path / "Processed_data/networks/networks_catchments_intersections"

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"

In [ ]:
jamaica_boundary_path = base_path / "Inputs/Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(f"Original Jamaica boundary CRS: {jamaica_boundary.crs}")

In [ ]:
# Path to your water GeoPackage
pipelines_NWC_gpkg_path = networks_folder / "water/pipelines_NWC.gpkg"
pipelines_layers = fiona.listlayers(pipelines_NWC_gpkg_path)
print("Available layers:", pipelines_layers)

In [ ]:
hydrobasins = base_path / "Processed_data/HydroBASINS_Level12_Clipped_Jamaica.shp"
hydrobasins = gpd.read_file(hydrobasins)
print(hydrobasins.crs)

In [ ]:
# Read the roads layers (edges and nodes) from the GeoPackage.
pipelines_edges = gpd.read_file(pipelines_NWC_gpkg_path, layer="edges")

pipelines_edges = pipelines_edges.to_crs(jamaica_metric_grid_crs)

In [ ]:
# === For Line Features (Road Edges) ===
# Perform an overlay (intersection) between the road edges and hydrobasins.
# This will split the roads by the hydrobasin boundaries.
pipelines_edges_overlay = gpd.overlay(pipelines_edges, hydrobasins, how="intersection")

# Optionally, calculate the length of each pipeline segment (assuming a projected CRS)
pipelines_edges_overlay["length"] = pipelines_edges_overlay.geometry.length

# Example aggregation: Sum of pipeline lengths by hydrobasin catchment (using HYBAS_ID)
pipelines_length_by_catchment = pipelines_edges_overlay.groupby("HYBAS_ID")["length"].sum().reset_index()
print("Pipelines Length by Catchment:")
print(pipelines_length_by_catchment)

# Rename the computed 'length' column to avoid conflicts with an existing column (case-insensitive conflict)
pipelines_edges_overlay = pipelines_edges_overlay.rename(columns={"length": "computed_length"})

# Example aggregation: Sum of road lengths by hydrobasin catchment (using HYBAS_ID)
pipelines_length_by_catchment = pipelines_edges_overlay.groupby("HYBAS_ID")["computed_length"].sum().reset_index()
print("Pipelines Length by Catchment:")
print(pipelines_length_by_catchment)

In [ ]:
# Define the output file path for the pipeline edges layer
pipelines_edges_catchments_intersection = networks_catchments_intersections / "pipelines_edges_catchments_intersection.gpkg"

# Save the intersected edges layer to the new GeoPackage file
pipelines_edges_overlay.to_file(pipelines_edges_catchments_intersection, layer="intersected_edges", driver="GPKG")